# S6E11 | Grandmaster Ladder | Public LB **0.95330**

Frozen public-LB submission (**Kaggle ref `58151250`**) that held **#1** on the leaderboard before being edged to **#2**.

This notebook ships the exact scored file and a short write-up of the climb. It does **not** retrain the full private stack in-kernel (GPT-2 medium 5-fold, Part-12 wide LM, 7-model orthogonal harvest, SoftPseudo family). Those stages were built locally; the final ladder step is published here so the community can study the trajectory.

## Public LB ladder (our probes)

| Step | File / idea | Public LB |
|---|---|---:|
| Part-12 backbone | `v8` FSL-prop rect | 0.95307 |
| Slot 3 | Cand0 + F0/F2 contrastive | 0.95312 |
| Att 2 / Att 3 | LB-gradient ascent | 0.95315 → 0.95317 |
| Strike 2 | + homogeneous 5F orth (NADE/Sameer/kNN/P4/Graph) | 0.95319 |
| Strike 3A | + GLM-quad + graph GLM | 0.95322 |
| Strike 4A | continuation | 0.95323 |
| Strike 6 | reverse Strike5 tree dilution | 0.95325 |
| Strike 7 / 8 | continue high-slope axis | 0.95327 → **0.95329 (#1)** |
| **Strike 9** | slope-weighted 2D step | **0.95330 (#1)** |

## Method sketch (what actually moved LB)

1. **Rank-logit space** with PCHIP quantile mapping onto a fixed OOF reference.
2. **Honest 5-fold orthogonal residuals** only (LOFO OLS against the Part-12 / v8 control).
3. **LB-gradient continuation** with tiny Spearman distance (`1-ρ ≈ 1e-5`), not large reshuffles.
4. **Kill collinear tree bags** that diluted the LM backbone (Strike 5A: −0.00008 LB).

## Credits / public building blocks

Part-12 wide LM stack and fraud-series artifacts (**goodpjw2008**), Fraud Signal Lab (**Malyshev**), Sameer wide pool, megayak LM/shallow trees, and many public OOF banks. This notebook freezes *our* scored endpoint for reproducibility of the **0.95330** row.

> SHA-256 of the attached CSV: `f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105`

In [ ]:
from pathlib import Path
import hashlib
import shutil
import pandas as pd

EXPECTED_SHA = "f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105"
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/submission.csv")

def locate(name: str) -> Path:
    hits = sorted(INPUT.rglob(name))
    assert hits, f"missing {name} under {INPUT}"
    return hits[0]

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

src = locate("submission_095330.csv")
digest = sha256(src)
assert digest == EXPECTED_SHA, (digest, EXPECTED_SHA)

sample = pd.read_csv(locate("sample_submission.csv"))
chk = pd.read_csv(src)
assert list(chk.columns) == ["id", "is_fraud"]
assert len(chk) == len(sample)
assert (chk["id"].to_numpy() == sample["id"].to_numpy()).all()
assert chk["is_fraud"].between(0, 1).all()

# Byte-identical copy so Public LB matches ref 58151250 exactly
shutil.copyfile(src, OUT)
assert sha256(OUT) == EXPECTED_SHA
print("wrote", OUT)
print("rows", len(chk), "| mean", float(chk["is_fraud"].mean()), "| sha", digest)
chk.head()


In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.shape)
print(sub["is_fraud"].describe())
print(sub.head())

## How to use

1. **Commit & Run** this notebook (CPU is enough).
2. **Submit** the output `submission.csv` to [Predicting Credit Card Fraud](https://www.kaggle.com/competitions/playground-series-s6e11).
3. You should land on **Public LB 0.95330** (same file as ref `58151250`).

If this helps your research, an upvote is appreciated — and please upvote the public sources in the credits above.